In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

c:\Users\chris\Documents\Christian\Thesis\Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
print("Loading AutoTokenizer and AutoModelForTokenClassification...")
tokenizer = AutoTokenizer.from_pretrained("covalenthq/cryptoNER")
model = AutoModelForTokenClassification.from_pretrained("covalenthq/cryptoNER")

Loading AutoTokenizer and AutoModelForTokenClassification...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15677.43it/s]


In [ ]:
ner_pipeline = pipeline(
    "token-classification", 
    model=model, 
    tokenizer=tokenizer, 
    aggregation_strategy="simple"
)

In [ ]:
# ==========================================
# Complete Dictionary-based Entity Linking
# ==========================================
# Based on Top 10 Market Cap as of May 17, 2026
ASSET_DICTIONARY = {
    # 1. Bitcoin
    "BITCOIN": "asset_btc",
    "BTC": "asset_btc",
    
    # 2. Ethereum
    "ETHEREUM": "asset_eth",
    "ETH": "asset_eth",
    
    # 3. Tether
    "TETHER": "asset_usdt",
    "USDT": "asset_usdt",
    
    # 4. BNB
    "BNB": "asset_bnb",
    "BINANCE COIN": "asset_bnb",
    
    # 5. XRP
    "XRP": "asset_xrp",
    "RIPPLE": "asset_xrp",
    
    # 6. USDC
    "USD COIN": "asset_usdc",
    "USDC": "asset_usdc",
    
    # 7. Solana
    "SOLANA": "asset_sol",
    "SOL": "asset_sol",
    
    # 8. TRON
    "TRON": "asset_trx",
    "TRX": "asset_trx",
    
    # 9. Dogecoin
    "DOGECOIN": "asset_doge",
    "DOGE": "asset_doge",
    
    # 10. Hyperliquid
    "HYPERLIQUID": "asset_hype",
    "HYPE": "asset_hype"
}

In [ ]:
# ==========================================
# 3. The Extraction & Linking Function
# ==========================================
def extract_and_link_entities(news_text: str):
    """
    Extracts entities using XLM-RoBERTa and maps them to unique IDs.
    """
    raw_entities = ner_pipeline(news_text)
    
    linked_entities = {}
    
    for entity in raw_entities:
        # Clean the extracted word and uppercase it for dictionary matching
        word = entity['word'].replace(' ', '').strip().upper()
        confidence = round(float(entity['score']), 4)
        
        # Only process high-confidence extractions
        if confidence > 0.80:
            # Match the extracted word against our internal dictionary
            unique_id = ASSET_DICTIONARY.get(word)
            
            # If it exists in our system, link it!
            if unique_id and unique_id not in linked_entities:
                linked_entities[unique_id] = {
                    "asset_id": unique_id,
                    "matched_text": word,
                    "ner_confidence": confidence
                }
                
    return list(linked_entities.values())

In [ ]:
if __name__ == "__main__":
    sample_news_text = """
    Solana validators successfully deployed the new patch today. 
    Meanwhile, BTC dominance continues to hover around 52%, leaving 
    Ethereum and altcoins like HYPE struggling.
    """
    
    print("\n--- Processing News ---")
    print(sample_news_text.strip())
    
    print("\n--- Linked Entities for Frontend/Database ---")
    final_output = extract_and_link_entities(sample_news_text)
    
    for item in final_output:
        print(f"ID: {item['asset_id']} | Matched: {item['matched_text']} | Confidence: {item['ner_confidence']}")


--- Processing News ---
Solana validators successfully deployed the new patch today. 
    Meanwhile, BTC dominance continues to hover around 52%, leaving 
    Ethereum and altcoins like HYPE struggling.

--- Linked Entities for Frontend/Database ---
ID: asset_eth | Matched: ETHEREUM | Confidence: 0.9992
ID: asset_hype | Matched: HYPE | Confidence: 0.9487
